# RAPTOR：递归聚类摘要与层次检索

**面试问题：面对跨多篇文档的全局问题，RAPTOR 怎样构建树并保留证据链？**

## 回答主线

1. 平坦检索擅长找一两个最相似片段，却难回答“主要问题有哪些”这类跨簇全局问题。
2. RAPTOR 先把叶子 Chunk 向量聚类，再为每簇生成摘要节点，并可递归形成更高层树。
3. 查询既可以命中高层主题摘要，也可以沿命中节点展开到原始叶子证据。
4. 摘要只负责导航和聚合，最终结论必须由 descendant leaf 支撑。
5. 无来源摘要可能引入不存在的原因，因此每个摘要节点要保存 child_ids、版本和覆盖事实。
6. 评估应按问题类型比较主题覆盖、叶子 Recall、树访问数和引用正确性。

## 真实案例

八条客服事故分别属于支付、账号认证、物流和发票四类，每类两条。查询是“本周投诉的主要原因有哪些”，平坦 Top-2 最多覆盖两个主题；我们用词项向量、从零余弦聚类和抽取式摘要构建两层树，再展开证据。这是可读的离线教学实验，用来验证协议和算法，不能外推为线上模型收益。

### 输入预览：八条可读事故叶子

In [1]:
import math  # 导入平方根以计算余弦相似度。

leaves = [  # 构造四个主题、每类两条的事故记录。
    {"id": "L1", "text": "支付重复扣款，重试缺少幂等键", "topic": "支付"},  # 支付幂等事故。
    {"id": "L2", "text": "退款到账延迟，支付通道积压", "topic": "支付"},  # 支付通道事故。
    {"id": "L3", "text": "登录验证码过期，用户无法认证", "topic": "认证"},  # 验证码事故。
    {"id": "L4", "text": "风控误封账号，身份复核缓慢", "topic": "认证"},  # 风控认证事故。
    {"id": "L5", "text": "包裹轨迹三天不更新，承运商延迟", "topic": "物流"},  # 物流轨迹事故。
    {"id": "L6", "text": "仓库漏发配件，补寄处理缓慢", "topic": "物流"},  # 仓库履约事故。
    {"id": "L7", "text": "企业发票抬头错误，需要重开", "topic": "发票"},  # 发票字段事故。
    {"id": "L8", "text": "开票申请积压，超过承诺期限", "topic": "发票"},  # 开票时效事故。
]  # 完成叶子语料。
terms = ["支付", "扣款", "退款", "认证", "登录", "账号", "物流", "包裹", "仓库", "发票", "开票", "延迟", "错误"]  # 定义可解释主题词表。
def vectorize(text):  # 把文本映射为词项出现向量。
    return [1.0 if term in text else 0.0 for term in terms]  # 每维表示一个可读词项是否出现。

for leaf in leaves:  # 为每个叶子附加向量。
    leaf["vector"] = vectorize(leaf["text"])  # 保存检索和聚类表示。
print("叶子  主题  文本")  # 输出语料表头。
for leaf in leaves:  # 逐条展示事故和人工主题仅用于评估。
    print(f"{leaf['id']}    {leaf['topic']:<4} {leaf['text']}")  # 展示八条真实语义记录。

叶子  主题  文本
L1    支付   支付重复扣款，重试缺少幂等键
L2    支付   退款到账延迟，支付通道积压
L3    认证   登录验证码过期，用户无法认证
L4    认证   风控误封账号，身份复核缓慢
L5    物流   包裹轨迹三天不更新，承运商延迟
L6    物流   仓库漏发配件，补寄处理缓慢
L7    发票   企业发票抬头错误，需要重开
L8    发票   开票申请积压，超过承诺期限


## Baseline 基线：平坦 Top-2 只能覆盖局部主题

In [2]:
def cosine(left, right):  # 实现两个向量的余弦相似度。
    numerator = sum(a * b for a, b in zip(left, right))  # 计算点积。
    left_norm = math.sqrt(sum(value * value for value in left))  # 计算左向量范数。
    right_norm = math.sqrt(sum(value * value for value in right))  # 计算右向量范数。
    return numerator / (left_norm * right_norm) if left_norm and right_norm else 0.0  # 处理零向量并返回相似度。

global_query = "本周投诉主要原因：支付 认证 物流 发票"  # 构造需要覆盖四类主题的全局问题。
query_vector = vectorize(global_query)  # 向量化查询。
flat_ranking = sorted(leaves, key=lambda leaf: (-cosine(query_vector, leaf["vector"]), leaf["id"]))  # 对所有叶子做平坦排序。
flat_top = flat_ranking[:2]  # 模拟固定检索预算只取两个叶子。
flat_topics = {leaf["topic"] for leaf in flat_top}  # 统计 Top-2 覆盖的主题。
print("Flat Top-2：")  # 输出平坦检索结果。
for leaf in flat_top:  # 逐候选展示得分和主题。
    print(f"{leaf['id']} topic={leaf['topic']} score={cosine(query_vector, leaf['vector']):.3f} text={leaf['text']}")  # 展示排序平局导致覆盖不足。
print(f"主题覆盖={len(flat_topics)}/4 -> {sorted(flat_topics)}")  # 量化全局问题失败。

Flat Top-2：
L1 topic=支付 score=0.354 text=支付重复扣款，重试缺少幂等键
L3 topic=认证 score=0.354 text=登录验证码过期，用户无法认证
主题覆盖=2/4 -> ['支付', '认证']


### 核心实现：从零聚类、摘要节点和树索引

In [3]:
def average_vector(nodes):  # 计算一组子节点的质心向量。
    width = len(nodes[0]["vector"])  # 读取向量维度。
    return [sum(node["vector"][index] for node in nodes) / len(nodes) for index in range(width)]  # 逐维求平均。

def agglomerative_pairs(nodes, pair_count):  # 用最大余弦相似度贪心合并成固定数量的二元簇。
    remaining = list(nodes)  # 复制节点避免修改输入。
    clusters = []  # 收集合并后的节点对。
    while len(clusters) < pair_count:  # 直到得到期望簇数。
        best = None  # 保存当前最相似节点对。
        for left_index in range(len(remaining)):  # 枚举左节点位置。
            for right_index in range(left_index + 1, len(remaining)):  # 枚举不重复右节点位置。
                score = cosine(remaining[left_index]["vector"], remaining[right_index]["vector"])  # 计算节点相似度。
                candidate = (score, -left_index, -right_index, left_index, right_index)  # 构造稳定比较元组。
                if best is None or candidate > best:  # 当前节点对更相似时更新。
                    best = candidate  # 保存最佳候选。
        left_index, right_index = best[3], best[4]  # 读取最佳节点位置。
        pair = [remaining[left_index], remaining[right_index]]  # 形成一个二元簇。
        clusters.append(pair)  # 保存该簇。
        remaining = [node for index, node in enumerate(remaining) if index not in {left_index, right_index}]  # 从候选集合移除已合并节点。
    return clusters  # 返回所有叶子簇。

clusters = agglomerative_pairs(leaves, pair_count=4)  # 将八个事故聚成四个主题簇。
summary_nodes = []  # 构造第一层摘要节点。
for cluster_index, children in enumerate(clusters, start=1):  # 逐簇生成抽取式摘要。
    topics = sorted({child["topic"] for child in children})  # 提取该簇覆盖主题。
    summary_text = f"主题{'/'.join(topics)}：" + "；".join(child["text"] for child in children)  # 用原文拼接形成不幻觉摘要。
    summary_nodes.append({"id": f"S{cluster_index}", "text": summary_text, "topic": "/".join(topics), "children": [child["id"] for child in children], "vector": average_vector(children), "facts": [child["text"] for child in children]})  # 保存摘要、子节点和事实集合。
root = {"id": "ROOT", "text": "本周投诉主题总览：" + "、".join(node["topic"] for node in summary_nodes), "children": [node["id"] for node in summary_nodes], "vector": average_vector(summary_nodes)}  # 构建覆盖四簇的根节点。
print("RAPTOR 树：")  # 输出层次结构。
print(f"{root['id']} children={root['children']} text={root['text']}")  # 展示根主题摘要。
for node in summary_nodes:  # 逐摘要展示叶子 provenance。
    print(f"  {node['id']} children={node['children']} topic={node['topic']} text={node['text']}")  # 展示聚类结果。

RAPTOR 树：
ROOT children=['S1', 'S2', 'S3', 'S4'] text=本周投诉主题总览：支付、认证、物流、发票
  S1 children=['L1', 'L2'] topic=支付 text=主题支付：支付重复扣款，重试缺少幂等键；退款到账延迟，支付通道积压
  S2 children=['L3', 'L4'] topic=认证 text=主题认证：登录验证码过期，用户无法认证；风控误封账号，身份复核缓慢
  S3 children=['L5', 'L6'] topic=物流 text=主题物流：包裹轨迹三天不更新，承运商延迟；仓库漏发配件，补寄处理缓慢
  S4 children=['L7', 'L8'] topic=发票 text=主题发票：企业发票抬头错误，需要重开；开票申请积压，超过承诺期限


## 结果解读：先检索摘要再展开叶子证据

In [4]:
summary_ranking = sorted(summary_nodes, key=lambda node: (-cosine(query_vector, node["vector"]), node["id"]))  # 对第一层摘要节点排序。
selected_summaries = summary_ranking[:4]  # 全局问题在摘要层用四个槽覆盖四个主题。
leaf_by_id = {leaf["id"]: leaf for leaf in leaves}  # 建立叶子 ID 索引供展开。
expanded_leaves = [leaf_by_id[child_id] for node in selected_summaries for child_id in node["children"]]  # 沿摘要节点展开原始证据。
raptor_topics = {leaf["topic"] for leaf in expanded_leaves}  # 统计最终证据主题覆盖。
visited_nodes = 1 + len(selected_summaries) + len(expanded_leaves)  # 统计根、摘要和叶子访问数量。
print("摘要排名：")  # 输出查询到树节点的中间过程。
for node in summary_ranking:  # 逐摘要展示相似度和后代。
    print(f"{node['id']} score={cosine(query_vector, node['vector']):.3f} topic={node['topic']} children={node['children']}")  # 展示全局主题导航。
print("最终证据叶子：", [(leaf["id"], leaf["topic"]) for leaf in expanded_leaves])  # 展示答案仍由原始事故支撑。
print(f"Flat主题覆盖={len(flat_topics)}/4，RAPTOR主题覆盖={len(raptor_topics)}/4，访问节点={visited_nodes}")  # 对比全局覆盖。
print("解读：高层摘要提供主题地图，展开阶段恢复八条原文；不能把 ROOT 的一句话直接当最终证据。")  # 解释层次检索。

摘要排名：
S1 score=0.378 topic=支付 children=['L1', 'L2']
S2 score=0.289 topic=认证 children=['L3', 'L4']
S4 score=0.289 topic=发票 children=['L7', 'L8']
S3 score=0.000 topic=物流 children=['L5', 'L6']
最终证据叶子： [('L1', '支付'), ('L2', '支付'), ('L3', '认证'), ('L4', '认证'), ('L7', '发票'), ('L8', '发票'), ('L5', '物流'), ('L6', '物流')]
Flat主题覆盖=2/4，RAPTOR主题覆盖=4/4，访问节点=13
解读：高层摘要提供主题地图，展开阶段恢复八条原文；不能把 ROOT 的一句话直接当最终证据。


## 失败案例：摘要加入不存在的“数据库故障”

In [5]:
hallucinated_summary = {"id": "S_bad", "text": "支付投诉主要由数据库故障导致", "children": ["L1", "L2"], "facts": ["数据库故障"]}  # 构造没有叶子证据的生成式摘要。
descendant_text = " ".join(leaf_by_id[child_id]["text"] for child_id in hallucinated_summary["children"])  # 合并后代原文供支持性检查。
unsupported_facts = [fact for fact in hallucinated_summary["facts"] if fact not in descendant_text]  # 找出无法在后代定位的摘要事实。
unsafe_claims = [hallucinated_summary["text"]]  # 模拟直接相信摘要进入最终答案。
safe_claims = [] if unsupported_facts else unsafe_claims  # 证据门禁拒绝没有 descendant 支持的结论。
print(f"摘要={hallucinated_summary['text']} children={hallucinated_summary['children']}")  # 展示看似合理的高层结论。
print(f"后代原文={descendant_text} unsupported={unsupported_facts}")  # 展示证据中没有数据库故障。
print(f"盲用claims={unsafe_claims}，证据门禁claims={safe_claims}")  # 展示修正前后差异。
print("修正策略：摘要节点保存 child_ids 和抽取事实；最终每个 claim 必须落到至少一个叶子 span，否则降级为待验证假设。")  # 总结 provenance 门禁。

摘要=支付投诉主要由数据库故障导致 children=['L1', 'L2']
后代原文=支付重复扣款，重试缺少幂等键 退款到账延迟，支付通道积压 unsupported=['数据库故障']
盲用claims=['支付投诉主要由数据库故障导致']，证据门禁claims=[]
修正策略：摘要节点保存 child_ids 和抽取事实；最终每个 claim 必须落到至少一个叶子 span，否则降级为待验证假设。


### 生产边界与节点 Manifest

In [6]:
node_manifest = {"node_id": summary_nodes[0]["id"], "children": summary_nodes[0]["children"], "summary_method": "extractive-teaching-r1", "embedding_version": "term-vector-r1", "fact_count": len(summary_nodes[0]["facts"])}  # 构造摘要节点版本清单。
print("节点 Manifest：", node_manifest)  # 展示重建和审计所需字段。
print("生产替换点：真实 RAPTOR 还需语义 Embedding、GMM/软聚类、递归摘要模型、增量更新、树存储、查询路由和带引用的生成评测。")  # 明确词项聚类边界。

节点 Manifest： {'node_id': 'S1', 'children': ['L1', 'L2'], 'summary_method': 'extractive-teaching-r1', 'embedding_version': 'term-vector-r1', 'fact_count': 2}
生产替换点：真实 RAPTOR 还需语义 Embedding、GMM/软聚类、递归摘要模型、增量更新、树存储、查询路由和带引用的生成评测。


## 回归测试：最后只保护聚类覆盖、叶子证据与幻觉门禁

In [7]:
assert len(summary_nodes) == 4 and all(len(node["children"]) == 2 for node in summary_nodes)  # 验证八个叶子形成四个二元摘要簇。
assert all(len(set(leaf_by_id[child]["topic"] for child in node["children"])) == 1 for node in summary_nodes)  # 验证本例每个簇聚合同主题事故。
assert len(flat_topics) < len(raptor_topics) and raptor_topics == {"支付", "认证", "物流", "发票"}  # 验证层次展开覆盖全部四类。
assert {leaf["id"] for leaf in expanded_leaves} == set(leaf_by_id)  # 验证全局答案可追溯到全部八个原始叶子。
assert unsupported_facts == ["数据库故障"] and safe_claims == []  # 验证无证据摘要事实被门禁阻断。
print("回归测试通过：主题聚类、全局覆盖、叶子展开、provenance 和摘要幻觉门禁均成立。")  # 用少量断言总结层次检索合同。

回归测试通过：主题聚类、全局覆盖、叶子展开、provenance 和摘要幻觉门禁均成立。
